# Notebook 03 — Analisi Demografica e Popolazione
**Capstone: Aree Interne Italiane**

In [1]:
import os
os.chdir(r'C:\Users\tomma\OneDrive\Desktop\EPICODE\CAPSTONE_2')
print("Working directory:", os.getcwd())

Working directory: C:\Users\tomma\OneDrive\Desktop\EPICODE\CAPSTONE_2


In [2]:
import pandas as pd
import numpy as np
import sqlite3
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

RAW_DIR       = "data/raw/"
PROCESSED_DIR = "data/processed/"
OUTPUT_DIR    = "outputs/"
DB_PATH       = "sql/aree_interne.db"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Percorsi configurati OK")

Percorsi configurati OK


## 1. Caricamento Dati Base

In [3]:
df_comuni = pd.read_csv(PROCESSED_DIR + "comuni_master_clean.csv",
                        dtype={"codice_comune": str})
print(f"comuni_master_clean: {len(df_comuni)} righe")
df_comuni.head()

comuni_master_clean: 7894 righe


,Codice Regione,provincia,Codice Provincia (Storico)(1),Progressivo del Comune (2),codice_comune,Denominazione (Italiana e straniera),comune,Denominazione altra lingua,cod_ripartizione,ripartizione,...,Codice NUTS1 2024,Codice NUTS2 2024 (3),Codice NUTS3 2024,classe_snai,desc_snai,classe_snai_old,desc_snai_old,is_area_interna,tipo_area,macro_area
0,1,201,1,1,001001,Agliè,Agliè,NaN,1,Nord-ovest,...,ITC,ITC1,ITC11,C,C - Cintura,C,C - Cintura,0,Cintura,Nord
1,1,201,1,2,001002,Airasca,Airasca,NaN,1,Nord-ovest,...,ITC,ITC1,ITC11,C,C - Cintura,C,C - Cintura,0,Cintura,Nord
2,1,201,1,3,001003,Ala di Stura,Ala di Stura,NaN,1,Nord-ovest,...,ITC,ITC1,ITC11,E,E - Periferico,E,E - Periferico,1,Periferico,Nord
3,1,201,1,4,001004,Albiano d'Ivrea,Albiano d'Ivrea,NaN,1,Nord-ovest,...,ITC,ITC1,ITC11,C,C - Cintura,C,C - Cintura,0,Cintura,Nord
4,1,201,1,6,001006,Almese,Almese,NaN,1,Nord-ovest,...,ITC,ITC1,ITC11,C,C - Cintura,C,C - Cintura,0,Cintura,Nord


## 2. Caricamento Dati Popolazione 2023
*Fonte: Popolazione comuni 2023 ISTAT*


In [4]:
df_pop_storica = pd.read_csv(RAW_DIR + "popolazione_storica_comuni_2018_2023.csv",
                              sep="|", encoding="utf-8-sig", dtype={"ITTER107": str})

print("Colonne disponibili:", df_pop_storica.columns.tolist())
print(f"Righe totali: {len(df_pop_storica)}")
df_pop_storica.head()

Colonne disponibili: ['ITTER107', 'Territorio', 'TIPO_DATO_CENS_POP', 'Tipo dato', 'TIME', 'Seleziona periodo', 'Value', 'Flag Codes', 'Flags']
Righe totali: 224490


,ITTER107,Territorio,TIPO_DATO_CENS_POP,Tipo dato,TIME,Seleziona periodo,Value,Flag Codes,Flags
0,IT,Italia,NPHH_AV,famiglie al 31 dicembre,2018,2018,25717041.00,NaN,NaN
1,IT,Italia,NPHH_AV,famiglie al 31 dicembre,2019,2019,25851122.38,NaN,NaN
2,IT,Italia,NPHH_AV,famiglie al 31 dicembre,2020,2020,26205757.00,NaN,NaN
3,IT,Italia,NPHH_AV,famiglie al 31 dicembre,2021,2021,26206246.00,NaN,NaN
4,IT,Italia,NPHH_AV,famiglie al 31 dicembre,2022,2022,26400326.00,NaN,NaN


In [5]:
df_pop_2023 = df_pop_storica[
    (df_pop_storica["TIPO_DATO_CENS_POP"] == "RESPOP_AV") &
    (df_pop_storica["TIME"] == 2023) &
    (df_pop_storica["ITTER107"].str.match(r"^\d{6}$"))
].copy()
df_pop_2023 = df_pop_2023[["ITTER107", "Value"]].copy()
df_pop_2023.columns = ["codice_comune", "popolazione_2023"]
print(f"Comuni con dati 2023: {len(df_pop_2023)}")

df_pop_2026_raw = pd.read_csv(RAW_DIR + "popolazione_comuni_2026.csv",
                               dtype={"codice_comune": str})
df_pop_2026 = df_pop_2026_raw[["codice_comune", "popolazione_2026"]].copy()
print(f"Comuni con dati 2026: {len(df_pop_2026)}")
print(f"Popolazione totale Italia 2026: {df_pop_2026['popolazione_2026'].sum():,.0f}")

Comuni con dati 2023: 7900
Comuni con dati 2026: 7896
Popolazione totale Italia 2026: 58,942,828


## 3. JOIN tra Comuni e Popolazione
Unione dati della popolazione al dataset dei comuni tramite il codice comune.

In [6]:
df_merged = df_comuni.merge(
    df_pop_2023[["codice_comune", "popolazione_2023"]],
    on="codice_comune", how="left"
).merge(
    df_pop_2026[["codice_comune", "popolazione_2026"]],
    on="codice_comune", how="left"
)

# Variazione demografica % 2023→2026
df_merged["var_pop_pct"] = (
    (df_merged["popolazione_2026"] - df_merged["popolazione_2023"])
    / df_merged["popolazione_2023"] * 100
)

n_2023 = df_merged["popolazione_2023"].notna().sum()
n_2026 = df_merged["popolazione_2026"].notna().sum()
print(f"Comuni totali:           {len(df_merged)}")
print(f"Con popolazione 2023:    {n_2023}")
print(f"Con popolazione 2026:    {n_2026}")

df_merged.head()

Comuni totali:           7894
Con popolazione 2023:    7512
Con popolazione 2026:    7516


,Codice Regione,provincia,Codice Provincia (Storico)(1),Progressivo del Comune (2),codice_comune,Denominazione (Italiana e straniera),comune,Denominazione altra lingua,cod_ripartizione,ripartizione,...,classe_snai,desc_snai,classe_snai_old,desc_snai_old,is_area_interna,tipo_area,macro_area,popolazione_2023,popolazione_2026,var_pop_pct
0,1,201,1,1,001001,Agliè,Agliè,NaN,1,Nord-ovest,...,C,C - Cintura,C,C - Cintura,0,Cintura,Nord,2596.0,2584.0,-0.462250
1,1,201,1,2,001002,Airasca,Airasca,NaN,1,Nord-ovest,...,C,C - Cintura,C,C - Cintura,0,Cintura,Nord,3686.0,3698.0,0.325556
2,1,201,1,3,001003,Ala di Stura,Ala di Stura,NaN,1,Nord-ovest,...,E,E - Periferico,E,E - Periferico,1,Periferico,Nord,472.0,468.0,-0.847458
3,1,201,1,4,001004,Albiano d'Ivrea,Albiano d'Ivrea,NaN,1,Nord-ovest,...,C,C - Cintura,C,C - Cintura,0,Cintura,Nord,1617.0,1633.0,0.989487
4,1,201,1,6,001006,Almese,Almese,NaN,1,Nord-ovest,...,C,C - Cintura,C,C - Cintura,0,Cintura,Nord,6315.0,6231.0,-1.330166


In [7]:
df_merged = df_comuni.merge(
    df_pop[["codice_comune", "popolazione_2023"]],
    on="codice_comune",
    how="left"
)

n_matched   = df_merged["popolazione_2023"].notna().sum()
n_unmatched = df_merged["popolazione_2023"].isna().sum()

print(f"Comuni totali:        {len(df_merged)}")
print(f"Con popolazione:      {n_matched}")
print(f"Senza corrispondenza: {n_unmatched}")

df_merged.head()

NameError: name 'df_pop' is not defined

In [ ]:
df_merged.to_csv(PROCESSED_DIR + "comuni_con_popolazione.csv", index=False)
print("Salvato: comuni_con_popolazione.csv")

## 4. Aggiornamento Database SQLite
Aggiungo la colonna popolazione_2023, popolazione_2026 e var_pop_pct alla tabella comuni nel database.

In [ ]:
conn = sqlite3.connect(DB_PATH)
cur  = conn.cursor()

# Aggiunge le colonne se non esistono
for col, tipo in [("popolazione_2023","INTEGER"),("popolazione_2026","INTEGER"),("var_pop_pct","REAL")]:
    try:
        cur.execute(f"ALTER TABLE comuni ADD COLUMN {col} {tipo}")
        print(f"Colonna {col} aggiunta.")
    except Exception:
        print(f"Colonna {col} gia esistente.")

# Aggiorna i valori riga per riga
updated = 0
for _, row in df_merged.iterrows():
    p23  = int(row["popolazione_2023"])  if pd.notna(row["popolazione_2023"])  else None
    p26  = int(row["popolazione_2026"])  if pd.notna(row["popolazione_2026"])  else None
    vp   = round(float(row["var_pop_pct"]),4) if pd.notna(row["var_pop_pct"]) else None
    cur.execute(
        "UPDATE comuni SET popolazione_2023=?, popolazione_2026=?, var_pop_pct=? WHERE codice_comune=?",
        (p23, p26, vp, str(row["codice_comune"]))
    )
    updated += 1

conn.commit()
print(f"Aggiornati {updated} comuni nel database.")

# Verifica
check = pd.read_sql(
    "SELECT COUNT(*) as n FROM comuni WHERE popolazione_2026 IS NOT NULL", conn
).iloc[0,0]
print(f"Comuni con popolazione 2026 nel DB: {check}")

## 5. Analisi Demografica
### 5.1 Popolazione per Classe SNAI

In [ ]:
df_ai = df_merged[df_merged["is_area_interna"] == 1].copy()
df_non_ai = df_merged[df_merged["is_area_interna"] == 0].copy()

pop_totale   = df_merged["popolazione_2023"].sum()
pop_ai       = df_ai["popolazione_2023"].sum()
pop_non_ai   = df_non_ai["popolazione_2023"].sum()

print(f"Popolazione totale Italia 2023:  {pop_totale:,.0f}")
print(f"Popolazione Aree Interne:        {pop_ai:,.0f} ({100*pop_ai/pop_totale:.1f}%)")
print(f"Popolazione fuori Aree Interne:  {pop_non_ai:,.0f} ({100*pop_non_ai/pop_totale:.1f}%)")

In [ ]:
order = ["Polo", "Polo Intercomunale", "Cintura", "Intermedio", "Periferico", "Ultraperiferico"]
order_map = {v: i for i, v in enumerate(order)}

df_plot = df_merged[
    df_merged["tipo_area"].isin(order) &
    df_merged["popolazione_2023"].notna()
].copy()
df_plot["tipo_area"] = df_plot["tipo_area"].astype(str)

df_snai_pop = df_plot.groupby("tipo_area")["popolazione_2023"].agg(
    n_comuni="count", pop_totale="sum"
).reset_index()
df_snai_pop["pop_media"] = (df_snai_pop["pop_totale"] / df_snai_pop["n_comuni"]).round(0)
df_snai_pop["_ord"] = df_snai_pop["tipo_area"].map(order_map)
df_snai_pop = df_snai_pop.sort_values("_ord").drop(columns=["_ord"])

print(df_snai_pop.to_string(index=False))

labels    = [str(v) for v in df_snai_pop["tipo_area"].tolist()]
vals_tot  = [float(v) for v in df_snai_pop["pop_totale"].tolist()]
vals_med  = [float(v) for v in df_snai_pop["pop_media"].tolist()]
colors    = ["#1F4E79","#2E75B6","#5BA3D9","#9DC3E6","#C6DCF0","#E2EFF8"][:len(labels)]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].bar(labels, vals_tot, color=colors)
axes[0].set_title("Popolazione Totale per Tipo Area", fontsize=12, fontweight="bold", color="#1F4E79")
axes[0].set_ylabel("Popolazione 2023")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x/1e6:.1f}M"))
axes[0].tick_params(axis="x", rotation=30)

axes[1].bar(labels, vals_med, color=colors)
axes[1].set_title("Popolazione Media per Comune", fontsize=12, fontweight="bold", color="#1F4E79")
axes[1].set_ylabel("Abitanti medi")
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x:,.0f}"))
axes[1].tick_params(axis="x", rotation=30)

plt.suptitle("Distribuzione Demografica per Classe SNAI",
             fontsize=14, fontweight="bold", color="#1F4E79", y=1.02)
plt.tight_layout()
plt.show()

### 5.2 Popolazione nelle Aree Interne per Macro-area

In [ ]:
macro_valid = ["Nord", "Centro", "Sud e Isole"]

df_ai_clean = df_ai[
    df_ai["macro_area"].isin(macro_valid) &
    df_ai["popolazione_2023"].notna()
].copy()
df_ai_clean["macro_area"] = df_ai_clean["macro_area"].astype(str)

df_macro = df_ai_clean.groupby("macro_area")["popolazione_2023"].agg(
    n_comuni="count", pop_totale="sum"
).reset_index()
df_macro["pop_media"] = (df_macro["pop_totale"] / df_macro["n_comuni"]).round(0)

print(df_macro.to_string(index=False))

macro_labels = [str(v) for v in df_macro["macro_area"].tolist()]
macro_vals   = [float(v) for v in df_macro["pop_totale"].tolist()]
macro_colors = ["#1F4E79","#2E75B6","#9DC3E6"][:len(macro_labels)]

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(macro_labels, macro_vals, color=macro_colors)
ax.set_title("Popolazione nelle Aree Interne per Macro-area",
             fontsize=13, fontweight="bold", color="#1F4E79")
ax.set_ylabel("Popolazione 2023")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x/1e6:.1f}M"))

for bar, val in zip(bars, macro_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50000,
            f"{val/1e6:.2f}M", ha="center", va="bottom", fontsize=10, color="#1F4E79")

plt.tight_layout()
plt.show()

### 5.3 Comuni piu Popolosi tra gli Ultraperiferici (F)

In [ ]:
df_F = df_merged[(df_merged["classe_snai"] == "F") &
                  df_merged["popolazione_2023"].notna() &
                  df_merged["comune"].notna()].copy()
df_F["comune"]  = df_F["comune"].astype(str)
df_F["regione"] = df_F["regione"].astype(str)
df_F_sorted = df_F.sort_values("popolazione_2023", ascending=False).head(20)

print(f"Comuni ultraperiferici con dato pop: {len(df_F)}")
print(df_F_sorted[["comune","regione","popolazione_2023"]].to_string(index=False))

labels_F = (df_F_sorted["comune"] + " (" + df_F_sorted["regione"].str[:3] + ")").tolist()

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(labels_F,
        df_F_sorted["popolazione_2023"].tolist(), color="#1F4E79")
ax.set_title("Top 20 Comuni Ultraperiferici per Popolazione",
             fontsize=12, fontweight="bold", color="#1F4E79")
ax.set_xlabel("Abitanti 2023")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 6. Query SQL — Analisi Demografica
### Query 5 — Regioni con Maggior Popolazione in Aree Interne

In [ ]:
q5 = """
SELECT
    regione,
    COUNT(*) AS n_comuni_ai,
    SUM(popolazione_2023) AS pop_aree_interne,
    ROUND(AVG(popolazione_2023), 0) AS pop_media_comune
FROM comuni
WHERE is_area_interna = 1
  AND popolazione_2023 IS NOT NULL
GROUP BY regione
ORDER BY pop_aree_interne DESC
"""

df_q5 = pd.read_sql(q5, conn)
print(df_q5.to_string(index=False))

fig, ax = plt.subplots(figsize=(11, 5))
ax.barh(df_q5["regione"], df_q5["pop_aree_interne"], color="#2E75B6")
ax.set_xlabel("Popolazione nelle Aree Interne")
ax.set_title("Popolazione nelle Aree Interne per Regione",
             fontsize=13, fontweight="bold", color="#1F4E79")
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x/1e3:.0f}k"))
ax.invert_yaxis()
plt.tight_layout()
plt.show()

### Query 6 — Piccoli Comuni Ultraperiferici (sotto 500 abitanti)

In [ ]:
q6 = """
SELECT
    comune, regione, macro_area, popolazione_2023
FROM comuni
WHERE classe_snai = 'F'
  AND popolazione_2023 < 500
  AND popolazione_2023 IS NOT NULL
ORDER BY popolazione_2023 ASC
LIMIT 30
"""

df_q6 = pd.read_sql(q6, conn)
print(f"Comuni F con meno di 500 abitanti: {len(df_q6)}")
print(df_q6.to_string(index=False))

## 7. Andamento Demografico 2019–2026
Utilizziamo la serie storica ISTAT (Popolazione al 1 gennaio) per regioni e province.
I dati coprono gli anni 2019–2026 e ci permettono di vedere il trend recente di spopolamento.